<div class="alert alert-info" role="alert"
     style="font-size: 1.4em; font-weight: bold; padding: 15px; margin: 10px 0;
            text-align: center; background-color: #d9edf7; border-color: #bce8f1;
            color: #31708f; border-radius: 8px;">
     Backfill missing OLCI bands into S3 bucket
<div>

### Import libraries

In [1]:
import eumdac
import time
from pathlib import Path

import boto3
import botocore
import s3fs

### Credentials for `EUMDAC`

In [11]:
consumer_key    = '8jpdMuohSLSyH6ep32alvA0M0qga'
consumer_secret = 'ZD3xfyRsVA_CTkEugjtuJKiTzG4a'
token = eumdac.AccessToken((consumer_key, consumer_secret))
datastore = eumdac.DataStore(token)

### Connecting to `S3` bucket

In [12]:
# Connecting to S3 bucket
fs = s3fs.S3FileSystem(
    client_kwargs={
        'endpoint_url': 'https://s3.waw3-1.cloudferro.com',
        'region_name': 'waw3-1'
    },
    config_kwargs={'signature_version': 's3v4'},
    profile='laurenb'
)
# Credentials
S3_ACCESS_KEY   = "1a6dca6f5c0e41b898e67aaea126ee31"
S3_SECRET_KEY   = "dc4e8b90121f46d1a4d4aa416406d9d2"
S3_ENDPOINT_URL = "https://s3.waw3-1.cloudferro.com"

s3 = boto3.client('s3', endpoint_url=S3_ENDPOINT_URL,
                   aws_access_key_id=S3_ACCESS_KEY,
                   aws_secret_access_key=S3_SECRET_KEY)

S3_BUCKET_NAME = "olci_l1_fr"
local_root     = '/home/nandii/olci_sg/olci_l1_fr'

### Settings

In [16]:
COLLECTION_ID = 'EO:EUM:DAT:0409'
#("EO:EUM:DAT:0885") Reprocessed

# Oa08 and Oa17 are needed for AFAI only
radbands = ('Oa08_radiance', 'Oa09_radiance', 'Oa12_radiance', 'Oa16_radiance', 'Oa17_radiance')

required_files = {'xfdumanifest.xml', 'geo_coordinates.nc', 'qualityFlags.nc'} | {f'{b}.nc' for b in radbands}

CHUNK_SIZE  = 1024 * 1024
MAX_RETRIES = 2
RETRY_DELAY = 5

out_dir = Path('/home/nandii/olci_sg/olci_l1_fr')
out_dir.mkdir(parents = True, exist_ok = True)

### Inventory the `S3` bucket and list missing files per scene

In [17]:
# Zero-byte objects count as missing
present = {}
for key, info in fs.find(f'{S3_BUCKET_NAME}/', detail = True, refresh = True).items():
    parts = key.split('/')
    if len(parts) == 3 and info['size'] > 0:
        present.setdefault(parts[1], set()).add(parts[2])

missing = {scene: required_files - files for scene, files in present.items()}
missing = {scene: files for scene, files in missing.items() if files}

print(f"{len(present)} scenes in bucket, {len(missing)} with missing files")

5499 scenes in bucket, 1158 with missing files


### Fetch missing files from `EUMDAC`, push to `S3`, delete local copy

In [ ]:
def remote_object_matches(bucket, key, local_size):
    try:
        head = s3.head_object(Bucket = bucket, Key = key)
    except botocore.exceptions.ClientError as e:
        if e.response.get('Error', {}).get('Code') in ('404', 'NotFound'):
            return False
        raise
    return head['ContentLength'] == local_size

failures = {}

for scene_name, files_needed in missing.items():
    try:
        product = datastore.get_product(collection_id = COLLECTION_ID, product_id = scene_name)
        entries = [e for e in product.entries if e.split('/')[-1] in files_needed]
    except Exception as e:
        print(f"Product lookup failed: {scene_name}: {e}")
        failures[scene_name] = sorted(files_needed)
        continue

    # Files EUMDAC does not list for this product
    not_listed = files_needed - {e.split('/')[-1] for e in entries}
    if not_listed:
        print(f"Not listed in EUMDAC: {scene_name}: {sorted(not_listed)}")
        failures.setdefault(scene_name, []).extend(sorted(not_listed))

    for entry in entries:
        filename   = entry.split('/')[-1]
        local_path = out_dir / entry
        s3_key     = entry
        local_path.parent.mkdir(parents = True, exist_ok = True)

        downloaded = False
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                with product.open(entry = entry) as fsrc, open(local_path, 'wb') as fdst:
                    while chunk := fsrc.read(CHUNK_SIZE):
                        fdst.write(chunk)
                downloaded = True
                break
            except Exception as e:
                print(f"Download attempt {attempt}/{MAX_RETRIES} failed for {entry}: {e}")
                local_path.unlink(missing_ok = True)
                if attempt < MAX_RETRIES:
                    time.sleep(RETRY_DELAY)

        if not downloaded:
            failures.setdefault(scene_name, []).append(filename)
            continue

        local_size = local_path.stat().st_size
        uploaded   = False
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                s3.upload_file(str(local_path), S3_BUCKET_NAME, s3_key)
                uploaded = True
                break
            except Exception as e:
                print(f"Upload attempt {attempt}/{MAX_RETRIES} failed for {s3_key}: {e}")
                if attempt < MAX_RETRIES:
                    time.sleep(RETRY_DELAY)

        if uploaded and remote_object_matches(S3_BUCKET_NAME, s3_key, local_size):
            local_path.unlink()
        else:
            print(f"Upload failed or unverified, keeping local copy: {s3_key}")
            failures.setdefault(scene_name, []).append(filename)

    # Remove the local scene folder once it is empty
    scene_folder = out_dir / scene_name
    if scene_folder.exists() and not any(scene_folder.iterdir()):
        scene_folder.rmdir()

    if scene_name not in failures:
        print(f"Completed: {scene_name}")

Completed: S3A_OL_1_EFR____20240726T125531_20240726T125831_20240727T160926_0179_115_095_2700_MAR_O_NT_004.SEN3
Completed: S3A_OL_1_EFR____20240726T125831_20240726T130131_20240727T160946_0180_115_095_2880_MAR_O_NT_004.SEN3
Completed: S3A_OL_1_EFR____20240727T122920_20240727T123220_20240728T152829_0179_115_109_2700_MAR_O_NT_004.SEN3
Completed: S3A_OL_1_EFR____20240727T123220_20240727T123520_20240728T152848_0179_115_109_2880_MAR_O_NT_004.SEN3
Completed: S3A_OL_1_EFR____20240728T134409_20240728T134709_20240729T163051_0179_115_124_2700_MAR_O_NT_004.SEN3
Completed: S3A_OL_1_EFR____20240728T134709_20240728T135009_20240729T163110_0180_115_124_2880_MAR_O_NT_004.SEN3
Completed: S3A_OL_1_EFR____20240729T131758_20240729T132058_20240730T154900_0179_115_138_2700_MAR_O_NT_004.SEN3


### Summary of failures — rerun the notebook to retry these

In [10]:
print(f"{len(failures)} scenes with failures")
for scene, files in failures.items():
    print(scene, files)

1158 scenes with failures
S3A_OL_1_EFR____20240726T125531_20240726T125831_20240727T160926_0179_115_095_2700_MAR_O_NT_004.SEN3 ['Oa08_radiance.nc', 'Oa17_radiance.nc']
S3A_OL_1_EFR____20240726T125831_20240726T130131_20240727T160946_0180_115_095_2880_MAR_O_NT_004.SEN3 ['Oa08_radiance.nc', 'Oa17_radiance.nc']
S3A_OL_1_EFR____20240727T122920_20240727T123220_20240728T152829_0179_115_109_2700_MAR_O_NT_004.SEN3 ['Oa08_radiance.nc', 'Oa17_radiance.nc']
S3A_OL_1_EFR____20240727T123220_20240727T123520_20240728T152848_0179_115_109_2880_MAR_O_NT_004.SEN3 ['Oa08_radiance.nc', 'Oa17_radiance.nc']
S3A_OL_1_EFR____20240728T134409_20240728T134709_20240729T163051_0179_115_124_2700_MAR_O_NT_004.SEN3 ['Oa08_radiance.nc', 'Oa17_radiance.nc']
S3A_OL_1_EFR____20240728T134709_20240728T135009_20240729T163110_0180_115_124_2880_MAR_O_NT_004.SEN3 ['Oa08_radiance.nc', 'Oa17_radiance.nc']
S3A_OL_1_EFR____20240729T131758_20240729T132058_20240730T154900_0179_115_138_2700_MAR_O_NT_004.SEN3 ['Oa08_radiance.nc', 'Oa17_r